# Data Science Internship - First Round Evaluation

This notebook runs and demonstrates all three solutions. Everything runs **locally and for free**: the models are
served by [Ollama](https://ollama.com), and no paid API is used.

| Question | What it builds | Folder |
|---|---|---|
| Q1 | Chat agent that answers questions about the inventory Excel file by writing and running pandas code, searching the web for definitions, and summarising in plain English | `q1_excel_agent/` |
| Q2 | Document-aware support assistant over the HDFC Life forms with session memory, no repetition, topic-switch handling and section citations | `q2_support_bot/` |
| Q3 | Pipeline that classifies 12 identity/insurance documents, extracts fields with per-field confidence, and flags low-confidence fields for human review | `q3_doc_pipeline/` |

Models: `qwen3:8b` (agent/chat), `qwen2.5vl:7b` (vision/handwriting), `nomic-embed-text` (embeddings).
Setup instructions are in `README.md`. The outputs below were produced on an Apple M5 laptop with 16 GB RAM.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
from IPython.display import Markdown, display
ROOT = Path.cwd()
import ollama
print("Models available:", sorted(m.model for m in ollama.list().models))

Models available: ['nomic-embed-text:latest', 'qwen2.5vl:7b', 'qwen3:8b']


---
## Q1 - Excel data agent

**How it works.** The model (`qwen3:8b`, tool-calling) gets two tools:
- `run_python(code)` executes pandas code with the sheet preloaded as `df`;
- `web_search(query)` runs a DuckDuckGo search for definitions or context.

It loops (call a tool, read the result, call again) until it can answer, then writes a plain-English summary.
The prompt forces it to compute every number with code and to state only values that appear in the tool output.

**Chat interface:** `streamlit run q1_excel_agent/app.py` (it shows the code the agent ran under each answer).

In [2]:
sys.path.insert(0, str(ROOT / "q1_excel_agent"))
from agent import DF, chat
print(DF.shape)
DF.head()

(46, 8)


,Product ID,Product Name,Opening Stock,Purchase/Stock in,Number of Units Sold,Hand-In-Stock,Cost Price Per Unit (USD),Cost Price Total (USD)
0,P101,Laptop,50,20,10,60,1200,72000
1,P102,Monitor,40,15,5,50,500,25000
2,P103,Keyboard,60,25,15,70,50,3500
3,P104,Headphones,30,10,3,37,100,3700
4,P105,Smartphone,70,30,20,80,900,72000


In [3]:
questions = [
    "Which product has the highest total cost, and how much is it?",
    "What are the top 3 best-selling products by units sold?",
    "What does 'hand-in-stock' mean? Look it up, then tell me which product has the lowest hand-in-stock.",
    "Does Opening Stock + Purchase - Units Sold always equal Hand-In-Stock? Which products don't match?",
]
history = []
for q in questions:
    history.append({"role": "user", "content": q})
    answer, steps = chat(history)
    history.append({"role": "assistant", "content": answer})
    print("USER:", q)
    for s in steps:
        shown = s["input"].get("code") or s["input"].get("query")
        print(f"  [{s['tool']}] {shown}")
    print("AGENT:", answer, "\n" + "-" * 100)

USER: Which product has the highest total cost, and how much is it?
  [run_python] result = df.loc[df['Cost Price Total (USD)'].idxmax()]
result[['Product ID', 'Product Name', 'Cost Price Total (USD)']]
AGENT: The product with the highest total cost is **P101 (Laptop)**, with a total cost of **$72,000**. This is the highest value in the 'Cost Price Total (USD)' column. 
----------------------------------------------------------------------------------------------------


USER: What are the top 3 best-selling products by units sold?
  [run_python] df.sort_values(by='Number of Units Sold', ascending=False).head(3)[['Product ID', 'Product Name', 'Number of Units Sold']]
AGENT: The top 3 best-selling products by units sold are:  
1. **P105 (Smartphone)** with **20 units sold**  
2. **P114 (Ethernet Cable)** with **15 units sold**  
3. **P103 (Keyboard)** with **15 units sold**  

Note: There is a tie for 2nd and 3rd place, as both Ethernet Cable and Keyboard sold 15 units. 
----------------------------------------------------------------------------------------------------


USER: What does 'hand-in-stock' mean? Look it up, then tell me which product has the lowest hand-in-stock.
  [web_search] hand-in-stock inventory definition
  [run_python] result = df.loc[df['Hand-In-Stock'] == df['Hand-In-Stock'].min()]
result[['Product ID', 'Product Name', 'Hand-In-Stock']]
AGENT: The product with the lowest **Hand-In-Stock** is **P136 (External DVD Drive)** with **17 units**. This is the minimum value in the "Hand-In-Stock" column. 
----------------------------------------------------------------------------------------------------


USER: Does Opening Stock + Purchase - Units Sold always equal Hand-In-Stock? Which products don't match?
  [run_python] df['Calculated Hand-In-Stock'] = df['Opening Stock'] + df['Purchase/Stock in'] - df['Number of Units Sold']
mismatches = df[df['Hand-In-Stock'] != df['Calculated Hand-In-Stock']]
mismatches[['Product ID', 'Product Name', 'Hand-In-Stock', 'Calculated Hand-In-Stock']]
AGENT: The formula **Opening Stock + Purchase/Stock in - Number of Units Sold** should theoretically equal **Hand-In-Stock**. However, the data shows discrepancies for these products:

1. **P121 (Gaming Monitor)**: Actual 29 vs Calculated 31  
2. **P130 (CPU Cooler)**: Actual 34 vs Calculated 38  
3. **P131 (Monitor Stand)**: Actual 28 vs Calculated 32  
4. **P132 (Mouse Pad)**: Actual 58 vs Calculated 62  
5. **P134 (Cable Management Kit)**: Actual 34 vs Calculated 38  
6. **P135 (WiFi Adapter)**: Actual 23 vs Calculated 25  
7. **P136 (External DVD Drive)**: Actual 17 vs Calculated 19  
8. **P137 (Printe

Reference values for checking, computed directly with pandas:

In [4]:
df = DF
top = df["Cost Price Total (USD)"].max()
print("Highest total cost (ties included):", df[df["Cost Price Total (USD)"] == top][["Product ID", "Product Name", "Cost Price Total (USD)"]].values.tolist())
print("Top 3 sold:", df.nlargest(3, "Number of Units Sold")[["Product ID", "Product Name", "Number of Units Sold"]].values.tolist())
low = df["Hand-In-Stock"].min()
print("Lowest hand-in-stock (ties included):", df[df["Hand-In-Stock"] == low][["Product ID", "Product Name", "Hand-In-Stock"]].values.tolist())
gap = df["Opening Stock"] + df["Purchase/Stock in"] - df["Number of Units Sold"] - df["Hand-In-Stock"]
print("Rows where the stock formula does not balance:", df.loc[gap != 0, "Product ID"].tolist())

Highest total cost (ties included): [['P101', 'Laptop', 72000], ['P105', 'Smartphone', 72000]]
Top 3 sold: [['P105', 'Smartphone', 20], ['P103', 'Keyboard', 15], ['P114', 'Ethernet Cable', 15]]
Lowest hand-in-stock (ties included): [['P136', 'External DVD Drive', 17]]
Rows where the stock formula does not balance: ['P121', 'P130', 'P131', 'P132', 'P134', 'P135', 'P136', 'P137', 'P140', 'P141', 'P142', 'P146']


---
## Q2 - Support assistant with memory and citations

**Knowledge base:** the 8 HDFC Life onboarding forms supplied with Q3:
- Proposal declaration
- Assignment form
- NACH mandate
- FATCA annexure
- Benefit Illustration declaration
- Moral Hazard Questionnaire
- Multiple-policies consent
- Suitability Profiler declaration

The scans were transcribed into Markdown by the local vision model, then reviewed and corrected against the
scans (`q2_support_bot/kb/`), split by section heading, and embedded into a Chroma vector store.

**For each turn the assistant:**
1. Rewrites the question into a standalone search query using the conversation, which handles "it" and "that form" and topic switches.
2. Retrieves the 4 most relevant sections.
3. Answers from them, citing sources by ID; code then maps the IDs to exact section paths.
4. Updates an *already told* ledger. When a retrieved section was used before, the model is told so and must refer back instead of repeating.

**Chat interface:** `streamlit run q2_support_bot/app.py`.

In [5]:
os.chdir(ROOT / "q2_support_bot")
print(subprocess.run([sys.executable, "ingest.py"], capture_output=True, text=True).stdout)
os.chdir(ROOT)

indexed 63 sections from 8 documents



The full 12-turn demonstration (`q2_support_bot/demo_transcript.md`, produced by `python q2_support_bot/demo.py`):

In [6]:
display(Markdown((ROOT / "q2_support_bot" / "demo_transcript.md").read_text()))

# Q2 Demo Transcript (12 turns)

Generated by `python demo.py` with the local model. Each turn shows the user's message, what the turn is testing, the standalone search query the bot built from the conversation, and the reply with its section citation.

## Turn 1 - Start

**User:** Hi, I'm filling in my HDFC Life forms. What is the NACH mandate form for?

*Search query used:* `Hi, I'm filling in my HDFC Life forms. What is the NACH mandate form for?`

**Assistant:** The NACH mandate form authorizes HDFC Life to debit your account for premium payments via ECS/SI/DD/NACH, specifying details like account type, number, and frequency. It also outlines terms for handling failed transactions, premium changes, and tax adjustments.
(Source: NACH Mandate > Declaration - Clauses 1 to 6 (details, funds and authorisation); NACH Mandate > Important Note - Notes 8 to 13 (premium changes, maximum amount, PAN) (part 1); NACH Mandate > Mandate Details (filled in) (part 1); NACH Mandate > Declaration - Clauses 7 to 12 (changes, failed debits and taxes) (part 2))

## Turn 2 - Follow-up

**User:** Which debit frequency options does it offer?

*Search query used:* `What debit frequency options are available in the NACH mandate form for HDFC Life?`

**Assistant:** The NACH mandate form offers the following debit frequency options: Monthly, Quarterly (Qtly), Half-Yearly (H.Yrly), Yearly (Yrly), and As & when presented.
(Source: NACH Mandate > Mandate Details (filled in) (part 1))

## Turn 3 - Follow-up

**User:** What happens if the debit fails because there isn't enough money in my account?

*Search query used:* `What happens if the NACH debit fails due to insufficient funds in the account?`

**Assistant:** If the debit fails due to insufficient funds, HDFC Life may retry the transaction twice for ECS requests. For SI cases with HDFC Bank/Ratnakar Bank, the debit will be attempted again up to 180 days from the original debit date. No further details on additional charges or actions are provided in the documents.
(Source: NACH Mandate > Declaration - Clauses 7 to 12 (changes, failed debits and taxes) (part 2); NACH Mandate > Declaration - Clauses 13 to 21 (due dates, cancellation, revival) (part 2))

## Turn 4 - Topic switch

**User:** Different question - what is the FATCA annexure form and why do I need it?

*Search query used:* `What is the FATCA annexure form and why is it required?`

**Assistant:** The FATCA Annexure form is required for reporting under Section 285BA of the Income Tax Act, 1961, and related rules (114F-114H). It is filled by the proposer/policyholder/annuitant to provide details necessary for compliance with FATCA/CRS (Common Reporting Standard) obligations. This form helps HDFC Life determine if your account qualifies as a "Reportable Account" and ensures transparency for tax authorities in India and abroad.
(Source: FATCA Annexure > Purpose of the Annexure Form; FATCA Annexure > Terms and Conditions for the Proposer/Policyholder/Annuitant (part 1); FATCA Annexure > Terms and Conditions for the Proposer/Policyholder/Annuitant (part 2))

## Turn 5 - Follow-up

**User:** Which documents do I have to submit with it?

*Search query used:* `Which documents are required to be submitted with the FATCA Annexure form?`

**Assistant:** You need to submit a self-attested copy of either a Certificate of Residence, TIN, or Functional equivalent for all countries listed in the table.
(Source: FATCA Annexure > Section 2: Declaration of Tax Residency (part 2))

## Turn 6 - Repeat question

**User:** Sorry, what did you say the NACH form is for again?

*Search query used:* `What is the NACH form used for?`

**Assistant:** As I mentioned earlier, the NACH mandate form authorizes HDFC Life to debit your account for premium payments via ECS/SI/DD/NACH, specifying account details and terms for failed transactions, premium changes, and tax adjustments.
(Source: NACH Mandate > Declaration - Clauses 1 to 6 (details, funds and authorisation); NACH Mandate > Important Note - Notes 8 to 13 (premium changes, maximum amount, PAN) (part 2))

## Turn 7 - Topic switch

**User:** Now about the Moral Hazard Questionnaire: when do I need to fill it in?

*Search query used:* `When is the Moral Hazard Questionnaire required to be filled out?`

**Assistant:** You need to fill the Moral Hazard Questionnaire if you have nominated a person who does not fall under immediate family relationships (father, mother, spouse, children).
(Source: Moral Hazard Questionnaire > Purpose of the Questionnaire)

## Turn 8 - Follow-up

**User:** Will a nominee relationship written as 'others' be accepted?

*Search query used:* `Will a nominee relationship listed as "others" be accepted for the Moral Hazard Questionnaire?`

**Assistant:** The nominee relationship mentioned as "others" will not be accepted.
(Source: Moral Hazard Questionnaire > Questions About the Nominee (part 1); Moral Hazard Questionnaire > Explanation and Disclaimer)

## Turn 9 - Topic switch

**User:** What does assigning my policy mean, and does it affect my nominee?

*Search query used:* `What does assigning my policy mean and does it affect my nominee?`

**Assistant:** Assigning your policy transfers your rights, title, and interest in the life insurance policy to another person or institution (the assignee), such as a bank as security for a loan. This assignment automatically cancels any existing nomination made in the policy, **except** if the assignment is in favor of HDFC Life Insurance Company. Therefore, your nominee's rights will be affected unless the assignee is HDFC Life.
(Source: Policy Assignment Form > Important Notes on Assignment; Policy Assignment Form > What an Assignment Is)

## Turn 10 - Back-reference

**User:** Going back to the mandate form - can I cancel it later?

*Search query used:* `Can I cancel the mandate form later?`

**Assistant:** Yes, you can cancel the NACH mandate later by providing a written request to the bank or corporate entity where you authorized the debit. This is explicitly stated in the mandate details and the declaration clauses.
(Source: NACH Mandate > Mandate Details (filled in) (part 2))

## Turn 11 - Topic switch

**User:** What am I confirming when I sign the Benefit Illustration declaration?

*Search query used:* `What does signing the Benefit Illustration declaration confirm?`

**Assistant:** When you sign the Benefit Illustration declaration, you confirm that you have read, understood, and agree to the terms and conditions of the policy, the risks involved, and the product benefits outlined in the Benefit Illustration. This includes acknowledging the year-wise premium payments and benefits as detailed in the illustration.
(Source: Benefit Illustration Declaration > Customer Declaration and Signature; Benefit Illustration Declaration > What the Benefit Illustration Shows)

## Turn 12 - Memory

**User:** Can you list which forms we have discussed so far in this chat?

*Search query used:* `CONVERSATION`

**Assistant:** The forms discussed so far are: 
1. **NACH Mandate** (for premium payments via ECS/SI/DD/NACH). 
2. **FATCA Annexure** (for FATCA/CRS compliance). 
3. **Moral Hazard Questionnaire** (for nominees outside immediate family). 
4. **Policy Assignment Form** (transferring policy rights). 
5. **Benefit Illustration Declaration** (confirming policy terms and benefits).
(Source: NACH Mandate > Declaration - Clauses 1 to 6 (details, funds and authorisation); FATCA Annexure > Purpose of the Annexure Form; Moral Hazard Questionnaire > Purpose of the Questionnaire; Policy Assignment Form > Important Notes on Assignment; Benefit Illustration Declaration > Customer Declaration and Signature)


---
## Q3 - Document classification, extraction and flagging

**Pipeline** (`q3_doc_pipeline/run.py`), one document at a time, with the local vision model `qwen2.5vl:7b`:
1. **Classify.** Two independent signals: keyword rules on the printed titles and labels, and the vision model's
   own choice among the 10 types (or "Other"). When both agree, classification confidence is 0.99. The two
   unlisted PDFs (proposal form, assignment form) are classified as *Other* and sent to human review.
2. **Extract.** Printed cards get 2 reads (two prompt styles). Handwritten forms get 3 reads: full page, a zoomed
   2x crop of the filled-in region, and the zoomed crop read character by character.
3. **Score each field:**
   - agreement across reads;
   - format and checksum rules: PAN, IFSC, Aadhaar Verhoeff, DL pattern, passport MRZ check digits (ICAO 9303), valid dates;
   - a discount for handwriting;
   - cross-document consistency (the same application number, name, DOB, PAN, signing date and place appear on several documents).
4. **Flag** every field below **0.80**.

Running the whole pipeline takes about 4 minutes on the laptop.

In [7]:
r = subprocess.run([sys.executable, "q3_doc_pipeline/run.py"], capture_output=True, text=True)
print(r.stdout[-2500:])

Aadhar.png                                       -> Aadhaar Card                       (66s)
Assignment Ashok.pdf                             -> Other / Unknown                    (107s)
ChatGPT Image May 2, 2026, 03_43_11 PM.png       -> Driving Licence                    (71s)
ChatGPT Image May 2, 2026, 03_52_54 PM.png       -> Passport                           (90s)
ECS.jpeg                                         -> NACH / ECS Mandate                 (118s)
Fatca.jpeg                                       -> FATCA Annexure Form                (234s)
ID.png                                           -> PAN Card                           (65s)
Illustration.jpeg                                -> Benefit Illustration Declaration   (137s)
Moral.jpeg                                       -> Moral Hazard Questionnaire         (82s)
Proposal Ashok.pdf                               -> Other / Unknown                    (17s)
split.jpeg                                       -> Multiple Polic

In [8]:
import pandas as pd
results = json.loads((ROOT / "q3_doc_pipeline/outputs/all_results.json").read_text())
rows = []
for d in results:
    for f, info in d["fields"].items():
        rows.append({"file": d["file"], "type": d["document_type"], "field": f, "value": info["value"],
                     "confidence": info["confidence"], "source": info["source"],
                     "flagged": info["confidence"] < 0.80})
table = pd.DataFrame(rows)
pd.set_option("display.max_rows", 100, "display.max_colwidth", 60)
table

,file,type,field,value,confidence,source,flagged
0,Aadhar.png,Aadhaar Card,Aadhaar Number,123456789012,0.50,printed,True
1,Aadhar.png,Aadhaar Card,Full Name,Mr. Ashok,1.00,printed,False
2,Aadhar.png,Aadhaar Card,Date of Birth,18/12/1979,1.00,printed,False
3,Aadhar.png,Aadhaar Card,Address,"S/O Kumar, Kataia, West Bihar India - 841543",1.00,printed,False
4,"ChatGPT Image May 2, 2026, 03_43_11 PM.png",Driving Licence,DL Number,MH12 2021 0001234,1.00,printed,False
5,"ChatGPT Image May 2, 2026, 03_43_11 PM.png",Driving Licence,Name,MR. ASHOK,1.00,printed,False
6,"ChatGPT Image May 2, 2026, 03_43_11 PM.png",Driving Licence,Date of Issue,15/06/2021,1.00,printed,False
7,"ChatGPT Image May 2, 2026, 03_43_11 PM.png",Driving Licence,Valid Till,14/06/2041,1.00,printed,False
8,"ChatGPT Image May 2, 2026, 03_52_54 PM.png",Passport,Passport Number,X1234567,1.00,printed,False
9,"ChatGPT Image May 2, 2026, 03_52_54 PM.png",Passport,Date of Birth,18/12/1979,1.00,printed,False


Example of the structured JSON for one handwritten document (`q3_doc_pipeline/outputs/ECS.json`):

In [9]:
ecs = json.loads((ROOT / "q3_doc_pipeline/outputs/ECS.json").read_text())
ecs.pop("reads")   # the raw per-read values are kept in the file; hidden here for brevity
print(json.dumps(ecs, indent=2, ensure_ascii=False))

{
  "file": "ECS.jpeg",
  "document_type": "NACH / ECS Mandate",
  "classification_confidence": 0.99,
  "classification_reason": "keyword rule and vision model agree (3 keyword hit(s))",
  "pages": 1,
  "fields": {
    "Bank Account Number": {
      "value": "31004258912",
      "confidence": 0.95,
      "agreement": "3/3",
      "rule": "9-18 digits",
      "source": "handwritten",
      "alternatives": [],
      "reads": {
        "31004258912": 3
      }
    },
    "IFSC Code": {
      "value": "SBIN0227112",
      "confidence": 0.95,
      "agreement": "3/3",
      "rule": "matches IFSC format AAAA0XXXXXX",
      "source": "handwritten",
      "alternatives": [],
      "reads": {
        "SBIN0227112": 3
      }
    },
    "Bank Name": {
      "value": "State Bank of India",
      "confidence": 0.95,
      "agreement": "3/3",
      "rule": "no format rule",
      "source": "handwritten",
      "alternatives": [],
      "reads": {
        "State Bank of India": 3
      }
    },
    

### Flagging report (threshold, rationale, flagged fields)

In [10]:
display(Markdown((ROOT / "q3_doc_pipeline/outputs/flag_report.md").read_text()))

# Q3 Flagging Report

**Threshold: 0.80**

Why 0.80:
- A field only clears 0.80 when **every independent read agrees** (3/3 for handwritten forms, 2/2 for printed
  cards) **and** the value does not fail its format/checksum rule. A single dissenting read drops a handwritten
  field to about 0.68 and sends it to review.
- In insurance onboarding the costs are lopsided: a wrong account number, IFSC or PAN means a failed debit,
  a KYC rejection or a compliance issue, while a human check costs seconds. So the threshold is set to catch
  every field with *any* disagreement, accepting some false alarms.
- Going higher (e.g. 0.95) would flag every handwritten field (they are capped at 0.95 by design) and make the
  report useless; going lower (e.g. 0.60) would let 2-of-3 majority votes through, which is exactly where
  misread digits hide.

## How a confidence score is computed

Per field: `share of reads that agree` +0.05 if it passes its format/checksum rule, capped at 0.50 if it fails, x0.95 if handwritten. Then a cross-document step compares values that must match across the customer's documents (application number, name, DOB, PAN, signing date/place): +0.05 when it agrees with the consensus; if it disagrees but an alternative read or a 1-character variant matches, the value is corrected and capped at 0.75/0.70 (so it is still reviewed); otherwise capped at 0.60.

## Summary

| File | Type | Fields | Flagged |
|---|---|---|---|
| Aadhar.png | Aadhaar Card | 4 | 1 |
| Assignment Ashok.pdf | Other / Unknown | 0 | 1 |
| ChatGPT Image May 2, 2026, 03_43_11 PM.png | Driving Licence | 4 | 0 |
| ChatGPT Image May 2, 2026, 03_52_54 PM.png | Passport | 4 | 1 |
| ECS.jpeg | NACH / ECS Mandate | 5 | 0 |
| Fatca.jpeg | FATCA Annexure Form | 5 | 1 |
| ID.png | PAN Card | 4 | 1 |
| Illustration.jpeg | Benefit Illustration Declaration | 4 | 3 |
| Moral.jpeg | Moral Hazard Questionnaire | 5 | 1 |
| Proposal Ashok.pdf | Other / Unknown | 0 | 1 |
| split.jpeg | Multiple Policies Consent Form | 4 | 0 |
| suitability.jpeg | Suitability Profiler Declaration | 5 | 1 |

## Fields below threshold (0.80)

| File | Field | Value | Confidence | Why |
|---|---|---|---|---|
| Aadhar.png | Aadhaar Number | `123456789012` | 0.5 | reads agreed 2/2; rule: 12 digits but Verhoeff check digit fails (number cannot be a real Aadhaar) |
| ChatGPT Image May 2, 2026, 03_52_54 PM.png | MRZ Line 2 | `X1234567<7IND7912185M3001010<<<<<<<<<<<<<<08` | 0.5 | reads agreed 2/2; rule: date of birth check digit is 5, computed 2; date of expiry check digit is 0, computed 9 |
| Fatca.jpeg | TIN / PAN | `BPQPD3051R` | 0.6 | reads agreed 3/3; rule: matches PAN format AAAAA9999A; CONFLICT (PAN): documents disagree and no value is confirmed by two documents |
| ID.png | PAN Number | `ABCDE1234F` | 0.6 | reads agreed 2/2; rule: matches PAN format AAAAA9999A; CONFLICT (PAN): documents disagree and no value is confirmed by two documents |
| Illustration.jpeg | Application Number | `1500137601025` | 0.68 | reads agreed 0/3; other readings: ['1500137401025', '1500137601005']; rule: digits only; RECONCILED (application / policy number): read '1500137601005' is 1 character from ['1500137601025'] on other documents; corrected - verify |
| Illustration.jpeg | Date | `26/04/2026` | 0.73 | reads agreed 2/3; other readings: ['06/04/2026']; rule: valid date |
| Illustration.jpeg | Place | `West Bihar` | 0.63 | reads agreed 1/3; other readings: ['West Biba']; rule: no format rule; RECONCILED (signing place): reads disagreed; chose 'West Bihar' over 'West Biba' because other documents say ['West Bihar'] |
| Moral.jpeg | Date | `26/04/2026` | 0.73 | reads agreed 2/3; other readings: ['26/04/2006']; rule: valid date |
| suitability.jpeg | Application Number | `1500137601025` | 0.73 | reads agreed 2/3; other readings: ['1500131601025']; rule: digits only |

## Documents routed to human review as a whole

- **Assignment Ashok.pdf**: Document type is not one of the 10 expected types; no extraction schema - route to human review. (no known type matched by either keyword rule or vision model)
- **Proposal Ashok.pdf**: Document type is not one of the 10 expected types; no extraction schema - route to human review. (no known type matched by either keyword rule or vision model)

## Cross-document consistency issues

- **PAN**: `BPQPD3051R` in Fatca.jpeg (TIN / PAN) vs ['ABCDE1234F'] elsewhere
- **PAN**: `ABCDE1234F` in ID.png (PAN Number) vs ['BPQPD3051R'] elsewhere


### Handwritten vs printed text, and failure cases

In [11]:
display(Markdown((ROOT / "q3_doc_pipeline/handwriting_notes.md").read_text()))

# Handwritten vs printed text: approach and observed failures

All extraction runs locally with a 7B vision-language model (`qwen2.5vl:7b` via Ollama). Plain OCR engines
such as Tesseract do badly on cursive and boxed handwriting, so a VLM reads both kinds of text.
The difference is in how many times we read, where we look, and how much we trust the result.

## Printed documents (Aadhaar, PAN, Driving Licence, Passport)
- **2 reads** of the full card with two prompt styles: a plain "extract these fields" prompt and a
  "character-by-character" prompt.
- Clean printed text almost always gives 2/2 agreement, so the deciding signal is **format and checksum rules**:
  - PAN must match `AAAAA9999A`.
  - The Aadhaar number must pass its Verhoeff check digit.
  - The DL number must match the state/RTO/year/serial pattern.
  - The passport MRZ line 2 must pass all ICAO 9303 check digits.
- Printed fields are not discounted for handwriting risk.

## Handwritten forms (NACH, FATCA and the four HDFC declarations)
1. **Zoom on the filled-in region.** Each form type has a fixed layout. The pipeline crops the band of the page
   holding the handwritten entries and upscales it 2x. This was the biggest single accuracy gain: in an
   early run, a crop that stopped just short of the Moral Hazard date line produced dates like `01/01/2026`.
2. **3 independent reads:** the full page, the zoomed region, and the zoomed region with a character-by-character
   prompt. That prompt warns about the usual confusions (1/7, 0/O, 5/S, 2/Z, 8/B).
3. **Field-level majority vote.** Confidence starts as the share of reads that agree. Handwritten fields are
   multiplied by 0.95, because three reads by the same model can share the same mistake.
4. **Format rules** (IFSC `AAAA0XXXXXX`, PAN-format TIN, 9-18 digit account number, valid dates).
5. **Cross-document reconciliation.** The same customer fills in all the forms, so the application number, name,
   signing date and place should match across forms. A value confirmed with high confidence on two or more
   documents is the consensus. A disagreeing read is corrected to it only when one of its alternative reads
   matches, or when it is one character away. A corrected value is capped at 0.70-0.75, so it **stays in the
   review queue**. The correction speeds up the reviewer; it doesn't replace them.

## Failure cases observed
| Document | Field | What went wrong | Outcome |
|---|---|---|---|
| Benefit Illustration | Application Number | Zoomed reads gave `1500137601005` (2 read as 0) and the full-page read gave `1500137401025` (6 read as 4). No two reads gave the true value. | Corrected to `1500137601025` by cross-document consensus (1 character away), flagged at 0.68 |
| Benefit Illustration | Place | Two reads gave `West Biba` (cursive "har" read as "ba") | Third read `West Bihar` chosen via consensus, flagged at 0.63 |
| Benefit Illustration | Date | Full-page read gave `06/04/2026`; the overwritten "26" is messy | Majority `26/04/2026`, flagged at 0.73 |
| Moral Hazard | Date | One read gave `26/04/2006` (handwritten 2 read as 0) | Majority `26/04/2026`, flagged at 0.73 |
| Suitability Profiler | Application Number | One read gave `1500131601025` (7 read as 1) | Majority correct, flagged at 0.73 |
| Driving Licence (printed) | Name | First version appended the "S/W/D of" value (`MR. ASHOK KUMAR`) | Fixed by making the field instruction explicit |

## Flags that are not reading errors
Each of these was read correctly but still needs a human:
- **Aadhaar number `1234 5678 9012`** fails the Verhoeff checksum, so it cannot be a genuine Aadhaar number.
  The card is labelled as a sample.
- **Passport MRZ line 2**: the date-of-birth and expiry check digits printed on the card are wrong
  (5 should be 2, 0 should be 9). A document like this would be rejected in production.
- **PAN vs TIN**: the PAN card says `ABCDE1234F`, while the TIN handwritten on the FATCA form is `BPQPD3051R`.
  For an Indian resident the TIN is the PAN, so one of the two is wrong. Neither value is confirmed by a second
  document, so both are flagged.
- **Father's name** (noted by hand, not compared automatically): the PAN card says `KUMAR`, the FATCA form
  `Arjun Das Kumar`. The two are compatible, but the fuller FATCA version is the one to keep on file.

## Limits
- The fixed crop regions assume these HDFC form layouts. A new form version would need its region updated in
  `config.py`; the full-page read still works without it, just less accurately.
- A 7B local model is the weakest link on messy cursive. Agreement voting catches *disagreement*, not errors
  where all reads make the same mistake. Checksums and cross-document consistency cover that gap for
  identifiers but not for free text such as names and places.
